<a href="https://colab.research.google.com/github/Vigneshvar16/CSA0607-DAA/blob/main/Assignment_3_CO3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import random
import math

def maxmin(a, lo, hi, cnt):
    """Divide & conquer: returns (min, max) of a[lo..hi]; cnt[0] counts comparisons."""
    if lo == hi:
        return a[lo], a[lo]
    if hi == lo + 1:                      # base case: 1 comparison for 2 elements
        cnt[0] += 1
        return (a[lo], a[hi]) if a[lo] < a[hi] else (a[hi], a[lo])
    mid = (lo + hi) // 2
    mn1, mx1 = maxmin(a, lo, mid, cnt)
    mn2, mx2 = maxmin(a, mid + 1, hi, cnt)
    cnt[0] += 2                           # combine: compare mins, compare maxes
    return min(mn1, mn2), max(mx1, mx2)

def binary_search(a, t):
    """Returns (index or -1, comparisons)."""
    lo, hi, c = 0, len(a) - 1, 0
    while lo <= hi:
        mid = (lo + hi) // 2
        c += 1
        if a[mid] == t: return mid, c
        c += 1
        if a[mid] < t: lo = mid + 1
        else: hi = mid - 1
    return -1, c

print("n      observed  ceil(3n/2)-2  naive 2(n-1)")
for n in (8, 100, 1000, 10000, 100000):
    a = random.sample(range(10 * n), n); c = [0]
    mn, mx = maxmin(a, 0, n - 1, c)
    assert mn == min(a) and mx == max(a)
    print(f"{n:<6} {c[0]:<9} {math.ceil(1.5*n)-2:<13} {2*(n-1)}")

print("\nBinary search (worst case ~ 2*(floor(log2 n)+1) counting both tests)")
print("n       found-avg  not-found  floor(log2 n)+1")
for n in (10, 1000, 100000, 1000000):
    a = list(range(0, 2 * n, 2))                      # even numbers only
    found = sum(binary_search(a, random.choice(a))[1] for _ in range(1000)) / 1000
    nf = binary_search(a, -1)[1]                      # unsuccessful search
    print(f"{n:<7} {found:<10.1f} {nf:<10} {int(math.log2(n))+1}")


n      observed  ceil(3n/2)-2  naive 2(n-1)
8      10        10            14
100    162       148           198
1000   1510      1498          1998
10000  15902     14998         19998
100000 165534    149998        199998

Binary search (worst case ~ 2*(floor(log2 n)+1) counting both tests)
n       found-avg  not-found  floor(log2 n)+1
10      4.8        6          4
1000    16.8       18         10
100000  30.5       32         17
1000000 36.8       38         20


In [3]:
import random, sys
sys.setrecursionlimit(20000)

def merge_sort(a, st):
    """st: dict with comparisons and max depth."""
    def rec(a, d):
        st["depth"] = max(st["depth"], d)
        if len(a) <= 1: return a
        m = len(a) // 2
        L, R = rec(a[:m], d + 1), rec(a[m:], d + 1)
        out, i, j = [], 0, 0
        while i < len(L) and j < len(R):
            st["cmp"] += 1
            if L[i] <= R[j]: out.append(L[i]); i += 1
            else: out.append(R[j]); j += 1
        return out + L[i:] + R[j:]
    return rec(a, 1)

def quick_sort(a, st, splits=None, d=1):
    """Pivot strategy: LAST element. Order-preserving partition into < and >=."""
    st["depth"] = max(st["depth"], d)
    if len(a) <= 1: return a
    p, rest = a[-1], a[:-1]
    st["cmp"] += len(rest)                       # each element compared with the pivot once
    L = [x for x in rest if x < p]; R = [x for x in rest if x >= p]
    if splits is not None: splits.append((len(L), len(R)))
    return quick_sort(L, st, splits, d + 1) + [p] + quick_sort(R, st, splits, d + 1)

def balanced_input(lo, hi):            # median first -> last-element pivot is the median
    if lo > hi: return []
    mid = (lo + hi) // 2
    return balanced_input(lo, mid - 1) + balanced_input(mid + 1, hi) + [mid]

print("First partition sizes (left,right) for n=15:")
for name, data in (("balanced  ", balanced_input(0, 14)), ("sorted    ", list(range(15)))):
    sp = []; quick_sort(data, {"cmp": 0, "depth": 0}, sp)
    print(name, sp[:4])

print("\nn=1000   algorithm / input        comparisons  max depth")
n = 1000
tests = {"random": random.sample(range(n), n), "balanced": balanced_input(0, n - 1),
         "sorted (worst)": list(range(n))}
for name, d in tests.items():
    s = {"cmp": 0, "depth": 0}; r = quick_sort(d, s); assert r == sorted(d)
    print(f"quick sort / {name:<15} {s['cmp']:<12} {s['depth']}")
    m = {"cmp": 0, "depth": 0}; assert merge_sort(d, m) == sorted(d)
    print(f"merge sort / {name:<15} {m['cmp']:<12} {m['depth']}")


First partition sizes (left,right) for n=15:
balanced   [(7, 7), (3, 3), (1, 1), (1, 1)]
sorted     [(14, 0), (13, 0), (12, 0), (11, 0)]

n=1000   algorithm / input        comparisons  max depth
quick sort / random          10947        23
merge sort / random          8722         11
quick sort / balanced        7987         10
merge sort / balanced        5497         11
quick sort / sorted (worst)  499500       1000
merge sort / sorted (worst)  4932         11


In [4]:
import random, time

def school(x, y, cnt):
    """Standard digit-by-digit multiplication: n*m single-digit products."""
    a, b = [int(c) for c in str(x)[::-1]], [int(c) for c in str(y)[::-1]]
    res = [0] * (len(a) + len(b))
    for i, da in enumerate(a):
        for j, db in enumerate(b):
            res[i + j] += da * db; cnt[0] += 1
    carry = 0
    for k in range(len(res)):
        carry, res[k] = divmod(res[k] + carry, 10)
    return int("".join(map(str, res[::-1])))

def karatsuba(x, y, cnt):
    if x < 10 or y < 10:                       # base case: single-digit product
        cnt[0] += 1; return x * y
    m = max(len(str(x)), len(str(y))) // 2
    p = 10 ** m
    a, b = divmod(x, p); c, d = divmod(y, p)   # x = a*p + b, y = c*p + d
    z2 = karatsuba(a, c, cnt)
    z0 = karatsuba(b, d, cnt)
    z1 = karatsuba(a + b, c + d, cnt) - z2 - z0   # 3 recursive products, not 4
    return z2 * p * p + z1 * p + z0

def timeit(f, x, y, reps=3):
    best = 1e9
    for _ in range(reps):
        c = [0]; t = time.perf_counter(); r = f(x, y, c); best = min(best, time.perf_counter() - t)
    return best, c[0], r

print("digits  school(ms) mults    karatsuba(ms) mults")
for n in (4, 8, 16, 32, 64, 128, 256, 512, 1024):
    x = random.randrange(10 ** (n - 1), 10 ** n); y = random.randrange(10 ** (n - 1), 10 ** n)
    t1, c1, r1 = timeit(school, x, y); t2, c2, r2 = timeit(karatsuba, x, y)
    assert r1 == r2 == x * y
    print(f"{n:<7} {t1*1000:<10.3f} {c1:<8} {t2*1000:<13.3f} {c2}")


digits  school(ms) mults    karatsuba(ms) mults
4       0.009      16       0.006         11
8       0.017      64       0.024         37
16      0.044      256      0.055         111
32      0.166      1024     0.198         381
64      0.625      4096     0.551         1015
128     2.452      16384    1.772         3281
256     10.297     65536    5.502         9755
512     45.741     262144   15.919        28815
1024    198.901    1048576  45.744        85633


In [5]:
import random, time

def add(A, B): return [[a + b for a, b in zip(r, s)] for r, s in zip(A, B)]
def sub(A, B): return [[a - b for a, b in zip(r, s)] for r, s in zip(A, B)]

def conventional(A, B, cnt):
    n = len(A); C = [[0] * n for _ in range(n)]
    for i in range(n):
        for k in range(n):
            a = A[i][k]
            for j in range(n):
                C[i][j] += a * B[k][j]; cnt[0] += 1
    return C

def strassen(A, B, cnt):
    n = len(A)
    if n == 1:
        cnt[0] += 1; return [[A[0][0] * B[0][0]]]
    h = n // 2
    sp = lambda M: ([r[:h] for r in M[:h]], [r[h:] for r in M[:h]], [r[:h] for r in M[h:]], [r[h:] for r in M[h:]])
    A11, A12, A21, A22 = sp(A); B11, B12, B21, B22 = sp(B)
    M1 = strassen(add(A11, A22), add(B11, B22), cnt)     # 7 products instead of 8
    M2 = strassen(add(A21, A22), B11, cnt)
    M3 = strassen(A11, sub(B12, B22), cnt)
    M4 = strassen(A22, sub(B21, B11), cnt)
    M5 = strassen(add(A11, A12), B22, cnt)
    M6 = strassen(sub(A21, A11), add(B11, B12), cnt)
    M7 = strassen(sub(A12, A22), add(B21, B22), cnt)
    C11 = add(sub(add(M1, M4), M5), M7); C12 = add(M3, M5)
    C21 = add(M2, M4);                   C22 = add(sub(add(M1, M3), M2), M6)
    return [a + b for a, b in zip(C11, C12)] + [a + b for a, b in zip(C21, C22)]

def multiply_any(A, B, cnt):
    """Pads to the next power of two with zeros, multiplies, then trims."""
    n = len(A); m = 1
    while m < n: m *= 2
    pad = lambda M: [r + [0] * (m - n) for r in M] + [[0] * m for _ in range(m - n)]
    C = strassen(pad(A), pad(B), cnt)
    return [r[:n] for r in C[:n]]

rnd = lambda n: [[random.randint(-9, 9) for _ in range(n)] for _ in range(n)]
def best_time(f, A, B, reps=3):
    t = 1e9
    for _ in range(reps):
        c = [0]; s = time.perf_counter(); r = f(A, B, c); t = min(t, time.perf_counter() - s)
    return t, c[0], r

print("n    conv mults  strassen mults  conv ms   strassen ms")
for n in (4, 8, 16, 32, 64):
    A, B = rnd(n), rnd(n)
    t1, c1, r1 = best_time(conventional, A, B); t2, c2, r2 = best_time(strassen, A, B)
    assert r1 == r2
    print(f"{n:<4} {c1:<11} {c2:<15} {t1*1000:<9.2f} {t2*1000:.2f}")

A, B = rnd(20), rnd(20); c = [0]                       # non power of two -> padded to 32
assert multiply_any(A, B, c) == conventional(A, B, [0])[:]
print("n=20 padded to 32: strassen mults =", c[0], "vs conventional 20^3 =", 20 ** 3)


n    conv mults  strassen mults  conv ms   strassen ms
4    64          49              0.02      0.22
8    512         343             0.10      1.47
16   4096        2401            0.79      9.48
32   32768       16807           6.07      70.96
64   262144      117649          54.26     635.99
n=20 padded to 32: strassen mults = 16807 vs conventional 20^3 = 8000


In [6]:
import random, math, time

def brute(P, cnt):
    best, pair = math.inf, None
    for i in range(len(P)):
        for j in range(i + 1, len(P)):
            cnt[0] += 1
            d = math.dist(P[i], P[j])
            if d < best: best, pair = d, (P[i], P[j])
    return best, pair

def closest_dc(P, cnt):
    Px = sorted(P)                                   # sorted by x once
    def rec(Px):
        n = len(Px)
        if n <= 3: return brute(Px, cnt)             # base case
        mid = n // 2; midx = Px[mid][0]
        dl, pl = rec(Px[:mid]); dr, pr = rec(Px[mid:])
        d, pair = (dl, pl) if dl <= dr else (dr, pr)  # best of the two halves
        strip = sorted((p for p in Px if abs(p[0] - midx) < d), key=lambda p: p[1])
        for i in range(len(strip)):                  # each point vs at most ~7 next in y-order
            for j in range(i + 1, len(strip)):
                if strip[j][1] - strip[i][1] >= d: break
                cnt[0] += 1
                dd = math.dist(strip[i], strip[j])
                if dd < d: d, pair = dd, (strip[i], strip[j])
        return d, pair
    return rec(Px)

print("n      brute dists  D&C dists  brute s   D&C s")
for n in (50, 200, 1000, 2000, 4000):
    P = [(random.uniform(0, 1000), random.uniform(0, 1000)) for _ in range(n)]
    c1, c2 = [0], [0]
    t = time.perf_counter(); d1, _ = brute(P, c1); t1 = time.perf_counter() - t
    t = time.perf_counter(); d2, _ = closest_dc(P, c2); t2 = time.perf_counter() - t
    assert abs(d1 - d2) < 1e-9
    print(f"{n:<6} {c1[0]:<12} {c2[0]:<10} {t1:<9.4f} {t2:.4f}")


n      brute dists  D&C dists  brute s   D&C s
50     1225         61         0.0005    0.0002
200    19900        256        0.0076    0.0010
1000   499500       1120       0.2059    0.0058
2000   1999000      2268       0.8611    0.0169
4000   7998000      4611       2.0248    0.0157


In [7]:
import random, bisect
from itertools import combinations

# ---------- Part A: k-th smallest (Median of Medians vs QuickSelect) ----------
def select_mom(a, k, st):
    """Deterministic worst-case O(n) selection; k is 0-based. st['cmp'] counts comparisons."""
    if len(a) <= 5:
        st["cmp"] += len(a) * len(a); return sorted(a)[k]
    meds = [sorted(a[i:i + 5])[len(a[i:i + 5]) // 2] for i in range(0, len(a), 5)]  # groups of 5
    pivot = select_mom(meds, len(meds) // 2, st)                                     # median of medians
    st["cmp"] += len(a)
    L = [x for x in a if x < pivot]; E = [x for x in a if x == pivot]
    if k < len(L): return select_mom(L, k, st)
    if k < len(L) + len(E): return pivot
    return select_mom([x for x in a if x > pivot], k - len(L) - len(E), st)

def quickselect(a, k, st, rule):
    """rule='first' (deterministic, poor on sorted data) or 'random'."""
    while True:
        if len(a) == 1: return a[0]
        p = a[0] if rule == "first" else random.choice(a)
        st["cmp"] += len(a)
        L = [x for x in a if x < p]; E = [x for x in a if x == p]
        if k < len(L): a = L
        elif k < len(L) + len(E): return p
        else: k -= len(L) + len(E); a = [x for x in a if x > p]

n = 2000
for name, data in (("random", random.sample(range(n), n)), ("sorted ", list(range(n)))):
    k = n // 2; ans = sorted(data)[k]; r = {}
    r["MoM"] = {"cmp": 0}; assert select_mom(data, k, r["MoM"]) == ans
    r["QS-first"] = {"cmp": 0}; assert quickselect(data, k, r["QS-first"], "first") == ans
    r["QS-random"] = {"cmp": 0}; assert quickselect(data, k, r["QS-random"], "random") == ans
    print(name, {m: v["cmp"] for m, v in r.items()})

# ---------- Part B: Meet-in-the-Middle subset sum ----------
def mitm(nums, target):
    h = len(nums) // 2
    def sums(part):
        return [sum(c) for r in range(len(part) + 1) for c in combinations(part, r)]
    A, B = sums(nums[:h]), sorted(sums(nums[h:]))      # 2^(n/2) sums per half
    for s in A:
        i = bisect.bisect_left(B, target - s)          # binary-search the complement
        if i < len(B) and B[i] == target - s: return True, len(A) + len(B)
    return False, len(A) + len(B)

def brute_subset(nums, target):
    cnt = 0
    for r in range(len(nums) + 1):
        for c in combinations(nums, r):
            cnt += 1
            if sum(c) == target: return True, cnt
    return False, cnt

nums = random.sample(range(1, 10000), 20); tgt = sum(nums) + 1     # worst case: no solution
print("\nn=20 target impossible: MITM", mitm(nums, tgt), " brute", brute_subset(nums, tgt))
t2 = sum(nums[:7]); assert mitm(nums, t2)[0] and brute_subset(nums, t2)[0]
print("n=40 MITM only needs", 2 * 2 ** 20, "stored/generated sums instead of", 2 ** 40, "subsets")


random {'MoM': 6502, 'QS-first': 5767, 'QS-random': 8626}
sorted  {'MoM': 6671, 'QS-first': 1501500, 'QS-random': 10341}

n=20 target impossible: MITM (False, 2048)  brute (False, 1048576)
n=40 MITM only needs 2097152 stored/generated sums instead of 1099511627776 subsets


In [8]:
import random

# Problem: Maximum subarray sum (largest sum of a contiguous block, e.g. best profit window).
def mss_dc(a, lo, hi, ops):
    """Returns (total, best_prefix, best_suffix, best_subarray) for a[lo..hi]."""
    if lo == hi:                                  # base case: one element
        return a[lo], a[lo], a[lo], a[lo]
    mid = (lo + hi) // 2
    t1, p1, s1, b1 = mss_dc(a, lo, mid, ops)
    t2, p2, s2, b2 = mss_dc(a, mid + 1, hi, ops)
    ops[0] += 1                                   # O(1) combine
    return (t1 + t2,
            max(p1, t1 + p2),                     # best prefix: left prefix or whole left + right prefix
            max(s2, t2 + s1),                     # best suffix
            max(b1, b2, s1 + p2))                 # best: in left, in right, or crossing the middle

def mss_brute(a):
    best, ops = a[0], 0
    for i in range(len(a)):
        s = 0
        for j in range(i, len(a)):
            s += a[j]; ops += 1
            best = max(best, s)
    return best, ops

# ---- correctness tests ----
tests = [[5], [-3], [-2, -1, -5], [1, 2, 3], [-2, 1, -3, 4, -1, 2, 1, -5, 4], [0, 0, 0]]
for t in tests:
    assert mss_dc(t, 0, len(t) - 1, [0])[3] == mss_brute(t)[0], t
for _ in range(500):
    t = [random.randint(-20, 20) for _ in range(random.randint(1, 60))]
    assert mss_dc(t, 0, len(t) - 1, [0])[3] == mss_brute(t)[0]
print("all correctness tests passed")

# ---- experiment ----
print("n       D&C combines  brute ops (n(n+1)/2)")
for n in (100, 1000, 5000):
    a = [random.randint(-50, 50) for _ in range(n)]; ops = [0]
    r = mss_dc(a, 0, n - 1, ops)[3]; b, bo = mss_brute(a); assert r == b


all correctness tests passed
n       D&C combines  brute ops (n(n+1)/2)
